# Triton Kernel 主线 · 第 5/10 课：行归约与 Padding 单位元

> 状态：**学习中（待提交）**  
> 本仓库采用逐课通过制。本课未通过前，不应直接进入下一课。

## 本课目标与完成标准

学完后你应能：实现非 2 次幂长度 row sum，解释 reduction 的 padding 单位元。

通过必须同时满足：

- 独立补齐本课唯一的代码填空题，并通过给定检查；
- 三个问答题均说明因果链，而不是只报术语；
- 能指出至少一个正确性边界和一个性能取舍；
- 总分不低于 8/10，且没有一票否决级概念错误。

## 前置关系

- 课程前置：Python、PyTorch 张量、CUDA 基本线程/内存概念
- 本课在路线中的作用：Triton `tl.sum` 在 program 内归约 tensor；常把 BLOCK 取为 N 的下一 2 次幂。

## 核心心智模型

### 1. 它是什么，解决什么问题

Triton `tl.sum` 在 program 内归约 tensor；常把 BLOCK 取为 N 的下一 2 次幂。

### 2. 它如何工作

mask 外 load 使用加法单位元 0，使 padding lanes 不改变 sum。

### 3. 正确性条件与常见误区

BLOCK 可能受硬件最大 block size 限制；N=0 无 next_power_of_2 的普通定义，应由 wrapper 拒绝或约定。

### 4. 性能与工程取舍

一 program 一行简单；超宽行需分块两阶段归约。

## 具体演示

N=100 时 BLOCK=128，128 lanes 中 100 个有效、28 个以 0 参与归约。

请在阅读后先合上这一节，用自己的语言复述“输入状态 → 中间状态 → 输出状态”，再做练习。

## 实践任务：唯一代码填空题

补齐 masked load 的 other。

规则：只能修改 `TODO`/`______` 所在位置；不要删除断言或放宽误差。代码注释说明了每个边界条件。

In [ ]:
import torch
import triton
import triton.language as tl

@triton.jit
def row_sum_kernel(x, out, N: tl.constexpr, stride_m: tl.constexpr,
                   BLOCK: tl.constexpr):
    row = tl.program_id(0)
    cols = tl.arange(0, BLOCK)
    mask = cols < N
    values = tl.load(x + row * stride_m + cols, mask=mask, other=______)  # TODO: 加法单位元
    tl.store(out + row, tl.sum(values, axis=0))

def row_sum(x):
    assert x.ndim == 2 and x.stride(1) == 1
    M, N = x.shape
    out = torch.empty((M,), device=x.device, dtype=x.dtype)
    row_sum_kernel[(M,)](x, out, N, x.stride(0), BLOCK=triton.next_power_of_2(N))
    return out

for shape in ((1, 1), (4, 100), (7, 257)):
    x = torch.randn(shape, device="cuda")
    torch.testing.assert_close(row_sum(x), x.sum(1), atol=1e-4, rtol=1e-4)


### 检查方法

在 CUDA/Triton 环境运行本单元格；断言覆盖规则尺寸和非规则尾块。首次 JIT 不计入性能。

提交时请给出：补齐后的代码、实际运行输出（环境不可用时注明“仅静态审查”）以及对失败用例的解释。

### Q1

不要背定义：请从输入、状态变化和输出三个阶段解释“行归约与 Padding 单位元”的工作机制。

**你的答案：**


### Q2

为什么 padding 有 28 lanes 而不是“只处理 100 lanes”？

**你的答案：**


### Q3

若归约是 max，other 应改成什么？

**你的答案：**


## 评分与通过规则

- 代码 4 分：正常输入 2 分，边界输入 1 分，解释实现 1 分；
- Q1～Q3 各 2 分；
- 一票否决：结果碰巧正确但核心因果链错误、删除边界检查、把未运行结果说成实测。

需要提示时按四级机制请求：概念区域 → 具体方向 → 关键局部 → 完整答案。

## 参考资料

- [Triton Tutorials](https://triton-lang.org/main/getting-started/tutorials/)
- [Triton language API](https://triton-lang.org/main/python-api/triton.language.html)

资料用于建立事实基线；面试回答仍需用自己的语言组织。